# 00 — Bootstrap

Run this first. It proves the machine is ready before any chapter runs, and it
fails **loudly and specifically** instead of leaving you guessing.

It checks, in order:

1. Python is **3.13** (3.14 cannot install `agentfield` at all)
2. `agentfield` imports and reports `0.1.132`
3. the control plane answers on `/health`
4. what is registered — and which of those nodes are *actually alive*
5. one real dispatch through the control plane
6. a workflow DAG, rendered as mermaid in this cell

### The mental model this repo is built on

The notebook is a **cockpit**, not a runtime. Agents run as processes registered
with the control plane; the notebook drives them **over HTTP**. That separation is
the whole point — and it is also load-bearing: driving an agent through
`await app.call(...)` from a notebook's own event loop is broken (it raises
`Timeout context manager should be used inside a task`, silently falls back, and
runs child reasoners **twice** while still returning a plausible-looking answer).
Everything here goes through the control plane instead.

In [1]:
import os, sys, json, time, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parent / "lib"))

OK, BAD, WARN = "\u2713", "\u2717", "!"
results = {}

def report(name, passed, detail=""):
    results[name] = bool(passed)
    print(f"{OK if passed else BAD} {name:<22} {detail}")
    return passed

def note(msg):
    print(f"{WARN} {msg}")

## 1 — Python 3.13

`agentfield==0.1.132` requires `>=3.10,<3.14`. On a 3.14 interpreter the install does not merely warn, it resolves to nothing and fails.

In [2]:
v = sys.version_info
report("python 3.13", v[:2] == (3, 13), f"{v.major}.{v.minor}.{v.micro}  ({sys.executable})")
if v[:2] != (3, 13):
    note("Wrong interpreter. Run `make setup`, then use the .venv kernel.")

✓ python 3.13            3.13.5  (/Users/santoshkumar/agentfield-workspaces/agentfield-8cead4e9/examples/blast-radius/.venv/bin/python)


## 2 — The SDK imports

In [3]:
try:
    import agentfield
    report("agentfield import", agentfield.__version__ == "0.1.132", agentfield.__version__)
except Exception as e:
    report("agentfield import", False, repr(e))

✓ agentfield import      0.1.132


## 3 — Configuration

The `openrouter/` prefix on `AI_MODEL` is a LiteLLM provider prefix and is required; the bare OpenRouter slug will not route.

In [4]:
env_path = pathlib.Path.cwd().parent / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, val = line.partition("=")
            os.environ.setdefault(k.strip(), val.strip())

SERVER = os.environ.get("AGENTFIELD_SERVER", "http://localhost:8080")
MODEL  = os.environ.get("AI_MODEL", "(unset)")
KEY    = os.environ.get("OPENROUTER_API_KEY", "")

report(".env present", env_path.exists(), str(env_path))
report("AGENTFIELD_SERVER", bool(SERVER), SERVER)
report("AI_MODEL prefixed", MODEL.startswith("openrouter/"), MODEL)
report("OPENROUTER_API_KEY", KEY.startswith("sk-or-") , "set" if KEY.startswith("sk-or-") else "MISSING or placeholder")
None


✓ .env present           /Users/santoshkumar/agentfield-workspaces/agentfield-8cead4e9/examples/blast-radius/.env
✓ AGENTFIELD_SERVER      http://localhost:8080
✓ AI_MODEL prefixed      openrouter/deepseek/deepseek-v4-flash
✓ OPENROUTER_API_KEY     set


## 4 — Control plane

In [5]:
import httpx

try:
    r = httpx.get(f"{SERVER}/health", timeout=5)
    h = r.json()
    report("control plane", r.status_code == 200 and h.get("status") == "healthy",
           f"{h.get('status')}  v{h.get('version')}")
except Exception as e:
    report("control plane", False, f"{e!r}")
    note("Start it with `make up` (or `af server`).")

✓ control plane          healthy  v1.0.0


## 5 — What is registered, and what is actually alive

Two separate questions. Registrations **outlive the process that made them**, so a
node whose process died still appears here looking healthy. Only the node's own
`/health` tells the truth, and skipping that check is how you end up debugging a
504 from a corpse.

(Do not use `af ls` for this: it truncates to 20 rows sorted by `last_run_at`, so a
freshly registered node that has never run is invisible and looks like a failure.)

In [6]:
import dag

try:
    caps = dag.nodes(SERVER)
    report("capabilities", True, f"{len(caps)} agent(s) registered")
    live = []
    for c in caps:
        aid = c.get("agent_id")
        if c.get("health_status") == "active" and dag.node_alive(aid, SERVER):
            live.append(c)
    print()
    dag.print_nodes(SERVER)
    print()
    report("at least one LIVE node", bool(live),
           ", ".join(c["agent_id"] for c in live) or "none answering their own /health")
except Exception as e:
    report("capabilities", False, repr(e))
    caps, live = [], []

✓ capabilities           9 agent(s) registered

   9 agent(s) registered:
     - brsmoke_nb [inactive] 2 reasoner(s) @ http://localhost:8055
     - brsmoke [active] 2 reasoner(s) @ http://localhost:8001
     - harness-duo-go [inactive] 3 reasoner(s) @ http://localhost:8317
     - twitter-creative-af [inactive] 13 reasoner(s) @ http://localhost:8017
     - swe-planner-go [inactive] 31 reasoner(s) @ http://localhost:8001
     - preprint-af [inactive] 19 reasoner(s) @ http://localhost:8012
     - preprint-af-go [inactive] 24 reasoner(s) @ http://localhost:8001
     - aforge [inactive] 1 reasoner(s) @ http://localhost:8001
     - market-signal-node [inactive] 11 reasoner(s) @ http://localhost:8001
   NOTE: registrations outlive the process. health_status can lie;
         dag.node_alive(<agent_id>) pings the node itself.

✓ at least one LIVE node brsmoke


## 6 — One real dispatch

Async dispatch, because it returns the `run_id` we need for the DAG.

Note the two different ids — mixing them costs hours:

| id | what takes it |
|---|---|
| `execution_id` | `GET /api/v1/executions/{id}` |
| `run_id` | `GET /api/v1/agentic/run/{id}`, `af wait` |

Passing a `run_id` to the executions endpoint returns *execution not found* forever.
`lib/dag.py` accepts either and resolves for you.

In [7]:
PREFERRED = "blast-radius"   # this repo's own node, once node/main.py is running
run_id = None

target_agent = next((c for c in live if c.get("agent_id") == PREFERRED), None) \
            or (live[0] if live else None)

if not target_agent:
    note("No live node to dispatch to. Start one with `make up`, then re-run.")
    note("Chapters 01+ bring up node/main.py; bootstrap tolerates its absence.")
    report("dispatch", False, "skipped - no live node")
else:
    aid = target_agent["agent_id"]
    reasoner = (target_agent.get("reasoners") or [{}])[0].get("id")
    # invocation_target uses a colon (agent:reasoner) but the execute endpoint
    # requires a dot (agent.reasoner). Passing the colon form returns HTTP 400.
    tgt = f"{aid}.{reasoner}"
    try:
        r = httpx.post(f"{SERVER}/api/v1/execute/async/{tgt}", json={"input": {}}, timeout=60)
        body = r.json()
        run_id = body.get("run_id") or body.get("workflow_id")
        # Any answer that came back FROM the node proves the round trip, even a
        # schema rejection: the control plane reached it and it replied.
        report("dispatch round-trip", r.status_code < 500, f"{tgt} -> HTTP {r.status_code}")
        if run_id:
            print(f"  run_id={run_id}")
        else:
            print(f"  no run_id in response: {json.dumps(body)[:200]}")
    except Exception as e:
        report("dispatch round-trip", False, repr(e))

✓ dispatch round-trip    brsmoke.A -> HTTP 202
  run_id=run_20260820_114700_cl93ua2t


## 7 — The DAG

This is the visual the whole talk is built around. `lib/dag.py` pulls
`GET /api/v1/agentic/run/{run_id}` and turns it into mermaid, rendered natively by
JupyterLab 4 — no CDN, no JavaScript.

The response is a **flat** list of executions; the tree is implicit in
`parent_execution_id`. There is no nested `children` array, and no cost or token
field anywhere in the payload.

In [8]:
target_run = run_id
if not target_run:
    try:
        recent = dag.recent_runs(SERVER)
        if recent:
            target_run = recent[0]["run_id"]
            note(f"No run of our own; rendering the most recent run on the control plane: {target_run}")
    except Exception as e:
        note(f"could not list recent runs: {e!r}")

out = None
if target_run:
    try:
        time.sleep(1.5)   # let the first executions land
        d = dag.fetch_run(target_run, SERVER)
        s = dag.stats(d)
        report("DAG fetched", True,
               f"{s['executions']} executions, depth {s['max_depth']}, fan-out {s['max_fanout']}")
        out = dag.render(target_run, server=SERVER, title=f"bootstrap run {target_run}")
    except Exception as e:
        report("DAG fetched", False, repr(e))
else:
    report("DAG fetched", False, "no run available to render")
out

✓ DAG fetched            1 executions, depth 1, fan-out 0


**bootstrap run run_20260820_114700_cl93ua2t** — 1 executions · depth 1 · max fan-out 0 · 1 agent(s)

```mermaid
flowchart TD
  n0["A<br/><small>✗ failed · 0.0s</small>"]
  class n0 bad;
  classDef ok   fill:#dcfce7,stroke:#16a34a,stroke-width:1px,color:#14532d;
  classDef run  fill:#dbeafe,stroke:#2563eb,stroke-width:1px,color:#1e3a8a;
  classDef wait fill:#f1f5f9,stroke:#94a3b8,stroke-width:1px,color:#334155;
  classDef bad  fill:#fee2e2,stroke:#dc2626,stroke-width:1px,color:#7f1d1d;
```

## 8 — Summary

In [9]:
print("bootstrap summary")
print("-" * 46)
for k, v in results.items():
    print(f"  {OK if v else BAD} {k}")
required = ["python 3.13", "agentfield import", "control plane"]
missing = [k for k in required if not results.get(k)]
print("-" * 46)
if missing:
    print(f"{BAD} NOT READY - fix: {', '.join(missing)}")
else:
    print(f"{OK} core environment is ready.")
    soft = [k for k, v in results.items() if not v]
    if soft:
        print(f"{WARN} non-blocking: {', '.join(soft)}")
        print("  (these need a running node - `make up` - and are expected to be")
        print("   red until node/main.py exists.)")

bootstrap summary
----------------------------------------------
  ✓ python 3.13
  ✓ agentfield import
  ✓ .env present
  ✓ AGENTFIELD_SERVER
  ✓ AI_MODEL prefixed
  ✓ OPENROUTER_API_KEY
  ✓ control plane
  ✓ capabilities
  ✓ at least one LIVE node
  ✓ dispatch round-trip
  ✓ DAG fetched
----------------------------------------------
✓ core environment is ready.


---

**Next:** `01_one_shot.ipynb` — one call, a schema, and a confidence flag.

TODO: chapter links once the chapters land.